In [1]:
import pennylane as qml
import numpy as np
import time

n_cells = [25, 25]
kx, ky, kz = (0.5, 0.6, 0.7)

t1 = time.time()
flat_hamiltonian = qml.spin.kitaev(n_cells, coupling=np.array([kx, ky, kz]))
flat_hamiltonian.compute_grouping()  # compute the qubit-wise commuting groups!

groups = []
for group_indices in flat_hamiltonian.grouping_indices:
    grouped_term = qml.sum(*(flat_hamiltonian.operands[index] for index in group_indices))
    groups.append(grouped_term)

grouped_hamiltonian = qml.sum(*groups)
t2 = time.time()
t_generation = t2 - t1

In [2]:
num_steps = 10
order = 6

@qml.qnode(qml.device("default.qubit"))
def executable_circuit(hamiltonian, num_steps, order):
    for wire in hamiltonian.wires: # uniform superposition over all basis states
        qml.Hadamard(wire)
    qml.TrotterProduct(hamiltonian, time=1.0, n=num_steps, order=order)
    return qml.state()

In [4]:
import pennylane.estimator as qre

t1 = time.time()
resources_exec = qre.estimate(executable_circuit)(grouped_hamiltonian, num_steps, order)
t2 = time.time()

print(f"Processing time: {(t2 - t1):.3g} seconds")
print(resources_exec)

Processing time: 18.3 seconds
--- Resources: ---
 Total wires: 1250
   algorithmic wires: 1250
   allocated wires: 0
     zero state: 0
     any state: 0
 Total gates : 2.972E+7
   'T': 2.670E+7,
   'CNOT': 1.214E+6,
   'Z': 6.000E+5,
   'S': 1.200E+6,
   'Hadamard': 1.250E+3
